In [27]:
import jax.numpy as jnp
import roughpy_jax as rpj
import numpy as np
from roughpy_jax.streams import LieIncrementStream
from roughpy_jax.streams.lie_increment_stream import _zero_lie
from roughpy_jax.intervals import IntervalType, Partition
from roughpy_jax.streams.piecewise_abelian_stream import to_piecewise_abelian_stream
from roughpy_jax.algebra import to_signature, antipode, to_log_signature, lie_to_tensor, as_free_tensor
from roughpy_jax.dense_algebra import get_batch_shape, _algebra_scalar_multiply, broadcast_to_batch_shape

In [28]:
# creating synthetic data and transforming it into an instance of a LieIncrementStream
 
def gen_linear_Lies(N, n, W, R):
    '''
    N is the no. of datapoints
    n is truncation level / degree of the log-n PDE
    W is the width of our stream (so the dimension our paths (X and Y) take values in)
    '''
    gen = np.linspace(0,1,N) 
    X = np.concatenate([(i+1)*gen.reshape(-1,1) for i in range(W)], axis=1) # array of N x W (representing a linear path in W dimensions with N timepoints)
    Y = np.concatenate([(2*i+1)*gen.reshape(-1,1) for i in range(W)], axis=1)

    X_inc = np.diff(X, axis=0) # incrementing the data for suitable transfer to LieIncrementStream
    Y_inc = np.diff(Y, axis=0) # since our original data is linear, in each dimension these are constant-valued
    times = np.delete(gen, 0) # deleting the first (!?) element of our gen array to get len(times) == len(X_inc)

    # creating Lie basis and Tensor basis 

    Lie_Basis = rpj.LieBasis(width = W, depth = n)
    Tensor_Basis = rpj.to_tensor_basis(Lie_Basis)

    # LieIncrementStream stores our data by calculating the log-signature of our data over (2^{resolution}) dyadic intervals, coarser dyadic intervals are also stored (so 2^{R+1}-1)
    # total log-signatures stored. For example say we have a path X, and we take resolution=1 then a log-signature will of X will be taken over [0,1/2) and [1/2, 1), this value is then
    # associated with the datapoint at the end of the interval (to make sure we aren't seeing into the future). 

    X_Lie = LieIncrementStream.from_increments(
                timestamps=times,
                data=X_inc,
                input_data_basis=None,
                resolution=R,
                lie_basis=Lie_Basis,
            )

    Y_Lie = LieIncrementStream.from_increments(
                timestamps=times,
                data=Y_inc,
                input_data_basis=None,
                resolution=R,
                lie_basis=Lie_Basis,
            )
    return X_Lie, Y_Lie, Lie_Basis, Tensor_Basis

In [29]:
# calculate log-signatures over each interval in a partition
def sigs_unif_intervals(X_Lie, Y_Lie, L, M):
    '''
    L - no. of uniform intervals to calculate the signature (and log-signature) of X over
    M - ... Y ...
    '''

    endpoints_X = jnp.linspace(0, 1, L + 1, dtype=jnp.float32).tolist()
    partition_X = Partition(endpoints_X, IntervalType.ClOpen)

    endpoints_Y = jnp.linspace(0, 1, M + 1, dtype=jnp.float32).tolist()
    partition_Y = Partition(endpoints_Y, IntervalType.ClOpen)


    X_LSP = tuple(lie_to_tensor(X_Lie.log_signature(interval)) for interval in partition_X.to_intervals())
    Y_LSP = tuple(lie_to_tensor(Y_Lie.log_signature(interval)) for interval in partition_Y.to_intervals())

    X_SP = tuple(rpj.ft_exp(x_ls, out_basis=x_ls.basis) for x_ls in X_LSP) 
    Y_SP = tuple(rpj.ft_exp(y_ls, out_basis=y_ls.basis) for y_ls in Y_LSP)

    return X_SP, Y_SP, X_LSP, Y_LSP

In [30]:
# truncating the log-signatures to depth n-1 
# we then change depth back to n, so that calculations still work (but the nth layer is now all zero)
def trunc(A_LSP, old_depth, new_depth):
    return tuple(a.change_depth(new_depth).change_depth(old_depth) for a in A_LSP)

In [31]:
def one_to_zero(X_SP, tensor_basis):

    X_sub = []
    
    for x in X_SP:
        x = jnp.array(x.__array__())
        x.at[0].set(0)
        x = rpj.FreeTensor(x, tensor_basis)
        X_sub.append(x)

    return X_sub

In [32]:
# setting up initial conditions for the PDE 
def initialise_PDE(X_SP, Y_SP, n, Tensor_Basis):

    L = len(X_SP)
    M = len(Y_SP)
    
    # K represents our target function f in the original PDE (Algorithm 5.1)
    K = np.zeros((L+1, M+1), dtype=np.float32) 

    # phi and psi follow notation from Algorithm 5.1 - batch_dims set to (1,) for now

    zero_tensor = rpj.FreeTensor.zero(basis=Tensor_Basis, batch_dims=(1,))
    phi = [[zero_tensor]*(M+1)]*(L+1)    
    psi = [[zero_tensor]*(M+1)]*(L+1)

    # Since the paper gives K[0, v] as the inner product of Z_0^x and Z_v^y and analogous for K[u, 0], I assume we are setting Z_0^x = 1 = Z_0^y where 1 = (1,0,0,0,...) 
    # is in the signature sense

    K[0,:] = 1. 
    K[:,0] = 1.

    # truncate signature and replace initial 1 with 0

    X_SPT = trunc(X_SP, n, n-1)
    Y_SPT = trunc(Y_SP, n, n-1)

    X_SPT_zero = one_to_zero(X_SPT, Tensor_Basis)
    Y_SPT_zero = one_to_zero(Y_SPT, Tensor_Basis)
    
    for i in range(1, L + 1):
        phi[i][0] = X_SPT_zero[i-1]
        
    for j in range(1, M + 1):
        psi[0][j] = Y_SPT_zero[j-1]        
    return K, phi, psi    

In [33]:
# Helper functions used to help calculations present in algorithm 5.1

def inner_prod(X,Y): # this function is created so we can get scalars instead of arrays as outputs (but probably slower so should fix later)
    return np.sum(X.__array__()*Y.__array__())  

def right_adj(A,C, tensor_basis):

    ant_A = antipode(A)
    ant_C = antipode(C)
    
    left_adj = rpj.ft_adjoint_left_mul(ant_A, ant_C)

    left_adj = rpj.FreeTensor(left_adj, basis=tensor_basis)  
    
    adj_A_C = antipode(left_adj)
    
    return adj_A_C

    
def eval_adj(phi, psi, x, y, tensor_basis):
    
    r_x_y = right_adj(x, y, tensor_basis)  
    r_y_x = right_adj(y, x, tensor_basis)

    return inner_prod(phi, r_x_y) + inner_prod(psi, r_y_x)

# unused func replaced by add_tensor_scalar
def tensor(scalar, batch_dims, basis, dtype):
    shape = batch_dims + (basis.size(),)
    t = np.zeros(dtype=jnp.dtype(dtype), shape=shape)
    t[0][0] = scalar
    return rpj.FreeTensor(t, basis)

def add_tensor_scalar(a, s):
    cls = type(a)
    scalar = jnp.asarray(s)
    ext_scalar = broadcast_to_batch_shape(scalar, a.batch_shape)
    result_data = jnp.add(a.data, ext_scalar)
    return cls(result_data, a.basis)

In [40]:
# lazy for loop implementation of algorithm 5.1 - change to JAX once working

def algorithm_solve(X_SP, Y_SP, X_LSP, Y_LSP, X_LSPT, Y_LSPT, K, phi, psi):

    L = len(X_LSP)
    M = len(Y_LSP)

    for i in range(L):
        for j in range(M):
                
            # phi (repl #+ rpj.ft_mul(phi[i][j+1], X_LSPT[i])\)
            phi[i+1][j+1] =  phi[i][j+1] + X_LSPT[i].__mul__(K[i,j])\
                            + rpj.ft_mul(phi[i][j+1], X_LSPT[i])\
                            + add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(psi[i][j+1], X_LSP[i])), -inner_prod(psi[i][j+1], X_LSPT[i]))

            # psi (repl #+ rpj.ft_mul(psi[i+1][j],Y_LSPT[j])\)
            psi[i+1][j+1] =  psi[i+1][j] + Y_LSPT[j].__mul__(K[i,j])\
                            + rpj.ft_mul(psi[i+1][j],Y_LSPT[j])\
                            + add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(phi[i+1][j], Y_LSP[j])), -inner_prod(phi[i+1][j], Y_LSPT[j]))

            eval_adj_ = eval_adj(phi[i][j], psi[i][j], X_LSP[i], Y_LSP[j], Tensor_Basis)
                

            # the kernel equation
            next_eval_adj = eval_adj(phi[i+1][j+1], psi[i+1][j+1], X_LSP[i], Y_LSP[j], Tensor_Basis)
            temp_2 =  eval_adj(phi[i][j+1], psi[i][j+1], X_LSP[i], Y_LSP[j], Tensor_Basis)
            temp_3 = eval_adj(phi[i+1][j], psi[i+1][j], X_LSP[i], Y_LSP[j], Tensor_Basis)
    
            G = inner_prod(X_LSP[i],Y_LSP[j])
            f_1 = K[i,j] * G + eval_adj_
            f_2 = K[i,j+1] * G + temp_2 
            f_3 = K[i+1,j] * G + temp_3 

            u_p = K[i+1,j] + K[i,j+1] - K[i,j] + f_1
            f_p = u_p * G + next_eval_adj


            K[i+1,j+1] =  K[i+1,j] + K[i,j+1] - K[i,j] + (1./4)*(f_1 + f_2 + f_3 + f_p)
    return K, phi, psi

In [41]:
def solve_PDE(X_Lie, Y_Lie, L, M, n, give_intermediate = False):

    X_SP, Y_SP, X_LSP, Y_LSP = sigs_unif_intervals(X_Lie=X_Lie, Y_Lie=Y_Lie, L=L, M=M)
    
    K_init, phi_init, psi_init = initialise_PDE(X_SP=X_SP, Y_SP=Y_SP, n=n, Tensor_Basis=Tensor_Basis)

    X_LSPT = trunc(X_LSP, n, n-1) 
    Y_LSPT = trunc(Y_LSP, n, n-1)

    K, phi, psi = algorithm_solve(X_SP=X_SP, Y_SP=Y_SP, X_LSP=X_LSP, Y_LSP=Y_LSP, X_LSPT=X_LSPT, Y_LSPT=Y_LSPT, K=K_init, phi=phi_init, psi=psi_init)
    if give_intermediate:
        return K, phi, psi
    else: 
        return K

In [64]:
N = 2500 # no. of datapoints
n = 5 # level of truncation / degree
W = 3 # width of stream (dimension of space it takes values in)
R = 12 # resolution
L = 36 # no. of intervals in partition of X
M = 36 # no. of ... X

X_Lie, Y_Lie, Lie_Basis, Tensor_Basis = gen_linear_Lies(N, n, W, R)

In [65]:
K = solve_PDE(X_Lie = X_Lie, Y_Lie = Y_Lie, L=L, M=M, n=n) # increasing R increases the final entry of K by quite a lot

In [66]:
x_sig = X_Lie.signature(X_Lie.support)
y_sig = Y_Lie.signature(Y_Lie.support)

k = rpj.tensor_pairing(x_sig, y_sig)
k

Array([1204.3634], dtype=float32)

In [67]:
K 

array([[1.0000000e+00, 1.0000000e+00, 1.0000000e+00, ..., 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00],
       [1.0000000e+00, 1.0252988e+00, 1.0473231e+00, ..., 1.6835546e+00,
        1.7048169e+00, 1.7206302e+00],
       [1.0000000e+00, 1.0582570e+00, 1.1098582e+00, ..., 2.8851862e+00,
        2.9533942e+00, 3.0044751e+00],
       ...,
       [1.0000000e+00, 2.0100825e+00, 3.3236468e+00, ..., 1.1185613e+03,
        1.2532952e+03, 1.3618732e+03],
       [1.0000000e+00, 2.0418866e+00, 3.4103329e+00, ..., 1.2550398e+03,
        1.4085026e+03, 1.5323433e+03],
       [1.0000000e+00, 2.0655341e+00, 3.4753008e+00, ..., 1.3651385e+03,
        1.5338821e+03, 1.6701896e+03]], shape=(37, 37), dtype=float32)